# CRS Inventory – Analysis Layers (Parcels, Census, Jobs)

This notebook inspects the coordinate reference systems (CRS) for the main
analysis layers, including:

- Parcel geometries
- Tract and/or block group geometries
- Any LODES spatial layers (if applicable)

**Goals:**
- Confirm the CRS for parcels.
- Confirm the CRS for census geometries (tracts, block groups).
- Identify any layers that are missing CRS information.
- Prepare for consistent reprojection to a single `PROJECT_CRS`.

In [1]:
from pathlib import Path
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

PARCELS_DIR = DATA_DIR / "parcels"
CENSUS_DIR = DATA_DIR / "census"
JOBS_DIR = DATA_DIR / "jobs"

PARCELS_DIR, CENSUS_DIR, JOBS_DIR

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs'))

In [5]:
analysis_files = {
    # Parcels
    "maricopa_parcels": PARCELS_DIR / "Parcels_20241108.shp",

    # Census
    "tracts_maricopa": CENSUS_DIR / "boundaries/US_tract_2023.shp",
    "blockgroups_maricopa": JOBS_DIR / "lodes_geometry/AZ_blck_grp_2023.shp",
}

analysis_files

{'maricopa_parcels': WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels/Parcels_20241108.shp'),
 'tracts_maricopa': WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census/boundaries/US_tract_2023.shp'),
 'blockgroups_maricopa': WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_geometry/AZ_blck_grp_2023.shp')}

In [6]:
def inspect_crs(layer_name: str, path: Path):
    """Load a spatial dataset and print its CRS."""
    if not path.exists():
        print(f"[MISSING] {layer_name}: {path}")
        return

    try:
        gdf = gpd.read_file(path)
        print(f"\n=== {layer_name} ===")
        print(f"Path: {path}")
        print(f"CRS: {gdf.crs}")
        print(f"Number of features: {len(gdf)}")
    except Exception as e:
        print(f"[ERROR] {layer_name}: {e}")

In [7]:
for name, path in analysis_files.items():
    inspect_crs(name, path)


=== maricopa_parcels ===
Path: c:\Users\arjav\DevSource\toc-performance-dashboard\data\parcels\Parcels_20241108.shp
CRS: EPSG:2868
Number of features: 1736192

=== tracts_maricopa ===
Path: c:\Users\arjav\DevSource\toc-performance-dashboard\data\census\boundaries\US_tract_2023.shp
CRS: PROJCS["USA_Contiguous_Albers_Equal_Area_Conic",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",37.5],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102003"]]
Number of features: 85074

==

## Draft project CRS decision (optional notes)

After inspecting both boundary and analysis layers, the tentatively finalized project CRS is EPSG:2868 (NAD83 Arizona Central). Local boundary shapefiles are already in this CRS and it will be more accurate to reproject shapefiles from the national level, rather than the other way around.

This CRS will be set in `00_project_setup.ipynb` as `PROJECT_CRS` once the decision is final.